# MyDigitalTwin — Apple
**Notebook 07 — Ingestion, exploration, nettoyage → Parquet**

Sources :
- `AppInstallActivity/App Install Activity.csv` → installations/mises à jour d'apps
- `AppleAccountInformation/Apps Using Sign In with Apple.csv` → apps connectées via Apple ID

Fichiers ignorés :
- `Apple Pay Cards.csv` / `Apple Card User Information.csv` → données bancaires
- `Apple ID SignOn Information.csv` → logs de connexion
- `Passkeys Information.csv` → sécurité
- `Calendar Preferences.csv` → 2 lignes de préférences, non analytique

Outputs :
- `data/parquet/apple_app_installs.parquet`
- `data/parquet/apple_signin_apps.parquet`

## Objectifs ML
- **K-Means (axe 3)** : activité temporelle (quand tu installes/mets à jour des apps)
- **ALS (axe 2)** : centres d'intérêt via les apps utilisées
- **Profil comportemental** : quelles apps tu utilises révèle tes habitudes

## 0. Initialisation

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Apple") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

APPLE_ROOT  = "../../data/raw/APPLE"
PARQUET_DIR = "../../data/parquet"

Spark version : 3.5.5


26/03/28 14:59:40 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE 1 — App Install Activity
### 1.1 Ingestion

In [2]:
RAW_PATH = f"{APPLE_ROOT}/AppInstallActivity/App Install Activity.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(RAW_PATH)

print(f"Lignes brutes : {df_raw.count():,}")
df_raw.printSchema()
df_raw.show(3, truncate=50)

Lignes brutes : 3,923
root
 |-- App Name: string (nullable = true)
 |-- Apple ID Number: string (nullable = true)
 |-- Application ID: string (nullable = true)
 |-- Application Type: string (nullable = true)
 |-- Client Event ID: string (nullable = true)
 |-- Created Date: string (nullable = true)
 |-- Device Identifier: string (nullable = true)
 |-- Device OS Version: string (nullable = true)
 |-- Event Date: string (nullable = true)
 |-- External Referral URL: string (nullable = true)
 |-- Installation Type: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OS Build Version: string (nullable = true)
 |-- Platform Name: string (nullable = true)
 |-- Store Front Name: string (nullable = true)

+--------+---------------+-----------------------+----------------+------------------------------------+------------------------+--------------------------------------------+-----------------+------------------------+---------------------+-----------------+------------------+---

In [3]:
# Colonnes supprimées (confidentialité / inutiles analytiquement)
DROP_COLS = [
    "Apple ID Number",       # identifiant personnel
    "Client Event ID",       # UUID interne
    "Device Identifier",     # identifiant appareil
    "External Referral URL", # inutile
    "OS Build Version",      # trop technique
    "Origin",                # toujours com.apple.AppStore
    "Store Front Name",      # toujours Belgium
    "Created Date",          # doublon de Event Date
]

df = df_raw.drop(*DROP_COLS)

# Parser la date (format ISO : 2025-02-02T05:00:00.000Z)
df = df.withColumn(
    "event_date",
    F.to_timestamp(F.col("Event Date"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
)

# Champs temporels
df = df \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

# Renommer les colonnes
df = df \
    .withColumnRenamed("App Name",           "app_name") \
    .withColumnRenamed("Application ID",     "app_id") \
    .withColumnRenamed("Application Type",   "app_type") \
    .withColumnRenamed("Device OS Version",  "ios_version") \
    .withColumnRenamed("Installation Type",  "install_type") \
    .withColumnRenamed("Platform Name",      "platform_device") \
    .withColumn("platform", F.lit("apple")) \
    .drop("Event Date")

print(f"Lignes : {df.count():,}")
df.printSchema()
df.show(5, truncate=50)

Lignes : 3,923
root
 |-- app_name: string (nullable = true)
 |-- app_id: string (nullable = true)
 |-- app_type: string (nullable = true)
 |-- ios_version: string (nullable = true)
 |-- install_type: string (nullable = true)
 |-- platform_device: string (nullable = true)
 |-- event_date: timestamp (nullable = true)
 |-- event_year: integer (nullable = true)
 |-- event_month: string (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- event_weekday: integer (nullable = true)
 |-- platform: string (nullable = false)

+---------------+--------------------------+--------+-----------+------------+---------------+-------------------+----------+-----------+----------+-------------+--------+
|       app_name|                    app_id|app_type|ios_version|install_type|platform_device|         event_date|event_year|event_month|event_hour|event_weekday|platform|
+---------------+--------------------------+--------+-----------+------------+---------------+-------------------+--------

### 1.2 Exploration

In [4]:
print("=== Répartition install vs update ===")
df.groupBy("install_type").count().orderBy(F.desc("count")).show()

print("\n=== Activité par année ===")
df.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Activité par jour de la semaine ===")
df.groupBy("event_weekday").count().orderBy("event_weekday").show()

=== Répartition install vs update ===


+----------------+-----+
|    install_type|count|
+----------------+-----+
|          update| 3100|
|   manual_update|  362|
|     auto_update|  182|
| initial_install|  132|
|       reinstall|  104|
|         restore|   41|
|   auto_download|    1|
|app_clip_install|    1|
+----------------+-----+


=== Activité par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2024| 1500|
|      2025| 1966|
|      2026|  457|
+----------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|  312|
|         1|  408|
|         2|  356|
|         3|  388|
|         4|  407|
|         5|  286|
|         6|  111|
|         7|  114|
|         8|   86|
|         9|  142|
|        10|  158|
|        11|  181|
|        12|  181|
|        13|  102|
|        14|   71|
|        15|   74|
|        16|   52|
|        17|   92|
|        18|   62|
|        19|   17|
+----------+-----+
only showing top 20 rows


=== Activité par jour de l

In [5]:
print("=== Top 30 apps les plus installées/mises à jour ===")
df.groupBy("app_name") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(30) \
    .show(truncate=50)

print("\n=== Apps distinctes ===")
print(f"  {df.select('app_name').distinct().count():,} apps distinctes")

print("\n=== Versions iOS utilisées ===")
df.groupBy("ios_version") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

=== Top 30 apps les plus installées/mises à jour ===
+------------------------------+-----+
|                      app_name|count|
+------------------------------+-----+
|                          NULL|  206|
|                             X|  134|
|                     Instagram|  118|
|                      Snapchat|  109|
|                       ChatGPT|  101|
|        Twitch: Live Streaming|   96|
|                     Messenger|   94|
|                       Netflix|   89|
|      Notion: Notes, Tasks, AI|   88|
|                        Google|   88|
|SoundCloud: The Music You Love|   88|
| Vinted: Shop & sell pre-loved|   86|
|                      Facebook|   86|
|                     Pinterest|   84|
|                       YouTube|   81|
|Discord - Talk, Play, Hang Out|   78|
|   Spotify: Music and Podcasts|   75|
| Temu: Shop Like a Billionaire|   70|
|                           bol|   70|
|       Gmail - Email by Google|   70|
+------------------------------+-----+
only showin

In [6]:
# Catégorisation des apps par nom
def categorize_app(col):
    name = F.lower(col)
    return (
        F.when(name.rlike(r"instagram|tiktok|twitter|snapchat|discord|whatsapp|messenger|telegram|threads"), "Réseaux sociaux")
         .when(name.rlike(r"spotify|deezer|soundcloud|rekordbox|djay|serato|traktor|shazam"), "Musique/DJ")
         .when(name.rlike(r"netflix|youtube|twitch|disney|prime video|canal"), "Streaming vidéo")
         .when(name.rlike(r"gmail|outlook|mail|calendar|notion|slack|teams|zoom|meet"), "Productivité")
         .when(name.rlike(r"uber|bolt|deliveroo|uber eats|just eat|takeaway"), "Transport/Livraison")
         .when(name.rlike(r"game|gaming|clash|pokemon|fifa|minecraft|pubg|fortnite"), "Jeux vidéo")
         .when(name.rlike(r"bank|banque|payconiq|paypal|revolut|wise|belfius|ing|bnp"), "Finance")
         .when(name.rlike(r"maps|google|safari|chrome|firefox|gps|waze"), "Navigation/Utilitaires")
         .when(name.rlike(r"photo|camera|lightroom|vsco|snapseed|capcut"), "Photo/Vidéo")
         .when(name.rlike(r"health|fitness|sport|nike|strava|myfitnesspal"), "Sport/Santé")
         .otherwise("Autre")
    )

df = df.withColumn("category", categorize_app(F.col("app_name")))

print("=== Répartition par catégorie ===")
df.groupBy("category") \
    .agg(F.count("*").alias("nb_events"),
         F.countDistinct("app_name").alias("nb_apps")) \
    .orderBy(F.desc("nb_events")) \
    .show()

=== Répartition par catégorie ===
+--------------------+---------+-------+
|            category|nb_events|nb_apps|
+--------------------+---------+-------+
|               Autre|     1874|    141|
|     Réseaux sociaux|      652|     10|
|             Finance|      341|     23|
|     Streaming vidéo|      270|      4|
|          Musique/DJ|      200|      4|
|Navigation/Utilit...|      179|     11|
|        Productivité|      173|      6|
|         Photo/Vidéo|      109|      6|
|         Sport/Santé|       65|      4|
|          Jeux vidéo|       37|      5|
| Transport/Livraison|       23|      3|
+--------------------+---------+-------+



### 1.3 Écriture Parquet

In [7]:
df.write.mode("overwrite").parquet(f"{PARQUET_DIR}/apple_app_installs.parquet")
print(f"✓ apple_app_installs.parquet — {df.count():,} lignes")

✓ apple_app_installs.parquet — 3,923 lignes


---
## PARTIE 2 — Sign In with Apple
### 2.1 Ingestion

In [8]:
SIGNIN_PATH = f"{APPLE_ROOT}/AppleAccountInformation/Apps Using Sign In with Apple.csv"

df_signin_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(SIGNIN_PATH)

print(f"Lignes brutes : {df_signin_raw.count():,}")
df_signin_raw.printSchema()
df_signin_raw.show(5, truncate=50)

Lignes brutes : 62
root
 |-- Developer: string (nullable = true)
 |-- Application: string (nullable = true)
 |-- Consented Date: string (nullable = true)
 |-- User ID: string (nullable = true)
 |-- Shared Info: string (nullable = true)
 |-- Forwarded Email: string (nullable = true)
 |-- Shared Email: string (nullable = true)
 |-- Consented Device: string (nullable = true)
 |-- Consented OS Version: string (nullable = true)
 |-- Sharing Indicator: string (nullable = true)

+----------------------+--------------+-----------------------+--------------------------------------------+-----------+-----------------------+-----------------------------------+----------------+--------------------+-----------------+
|             Developer|   Application|         Consented Date|                                     User ID|Shared Info|        Forwarded Email|                       Shared Email|Consented Device|Consented OS Version|Sharing Indicator|
+----------------------+--------------+----------

In [9]:
# Colonnes supprimées (confidentialité)
DROP_SIGNIN = [
    "User ID",          # identifiant unique Apple
    "Forwarded Email",  # email personnel
    "Shared Email",     # email masqué mais personnel
]

df_signin = df_signin_raw.drop(*DROP_SIGNIN)

# Parser la date (format : 2026-01-11 10:06:36.487)
df_signin = df_signin.withColumn(
    "consent_date",
    F.to_timestamp(F.col("Consented Date"), "yyyy-MM-dd HH:mm:ss.SSS")
)

# Champs temporels
df_signin = df_signin \
    .withColumn("event_year",  F.year("consent_date")) \
    .withColumn("event_month", F.date_format("consent_date", "yyyy-MM")) \
    .withColumnRenamed("Developer",          "developer") \
    .withColumnRenamed("Application",        "app_name") \
    .withColumnRenamed("Shared Info",        "shared_info") \
    .withColumnRenamed("Consented Device",   "device") \
    .withColumnRenamed("Consented OS Version", "os_version") \
    .withColumnRenamed("Sharing Indicator",  "sharing") \
    .withColumn("platform", F.lit("apple")) \
    .drop("Consented Date")

print(f"Lignes : {df_signin.count():,}")
df_signin.show(10, truncate=50)

Lignes : 62
+----------------------+---------------------------+-----------------+---------------+----------------+-------+-----------------------+----------+-----------+--------+
|             developer|                   app_name|      shared_info|         device|      os_version|sharing|           consent_date|event_year|event_month|platform|
+----------------------+---------------------------+-----------------+---------------+----------------+-------+-----------------------+----------+-----------+--------+
|          REDDIT, INC.|                     Reddit|            email|      iPhone 12|  iPhone OS 26.1|     No|2026-01-11 10:06:36.487|      2026|    2026-01|   apple|
| Rocky Road Games Inc.|                       SWAY|             name|      iPhone 12|  iPhone OS 26.1|     No|2025-12-13 19:28:31.112|      2025|    2025-12|   apple|
|      Best Secret GmbH|                 BestSecret|       name,email|      iPhone 12|  iPhone OS 18.5|     No|2025-08-04 20:24:53.564|      2025|  

### 2.2 Exploration

In [10]:
print("=== Apps connectées via Apple ID ===")
df_signin.select("app_name", "developer", "consent_date", "shared_info") \
    .orderBy(F.desc("consent_date")) \
    .show(truncate=50)

print("\n=== Info partagée avec les apps ===")
df_signin.groupBy("shared_info").count().orderBy(F.desc("count")).show()

print("\n=== Connexions par année ===")
df_signin.groupBy("event_year").count().orderBy("event_year").show()

=== Apps connectées via Apple ID ===
+-------------------------------+--------------------------------------------------+-----------------------+-----------------+
|                       app_name|                                         developer|           consent_date|      shared_info|
+-------------------------------+--------------------------------------------------+-----------------------+-----------------+
|                         Reddit|                                      REDDIT, INC.|2026-01-11 10:06:36.487|            email|
|                           SWAY|                             Rocky Road Games Inc.|2025-12-13 19:28:31.112|             name|
|                     BestSecret|                                  Best Secret GmbH|2025-08-04 20:24:53.564|       name,email|
|                 DuolingoMobile|                                     Duolingo, Inc|2025-05-27 08:49:43.904|       name,email|
|                         Viggle|                            WarpEngine Ca

### 2.3 Écriture Parquet

In [11]:
df_signin.write.mode("overwrite").parquet(f"{PARQUET_DIR}/apple_signin_apps.parquet")
print(f"✓ apple_signin_apps.parquet — {df_signin.count():,} lignes")

✓ apple_signin_apps.parquet — 62 lignes


---
## Résumé

In [12]:
print("=" * 58)
print("  MyDigitalTwin — Apple — Résumé")
print("=" * 58)
print(f"  App installs : {df.count():>8,} → apple_app_installs.parquet")
print(f"  Sign In apps : {df_signin.count():>8,} → apple_signin_apps.parquet")
print("=" * 58)
print()
print("  Usage ML :")
print("  - App installs → K-Means temporel (axe 3)")
print("  - App installs → centres d'intérêt ALS (axe 2)")
print("  - Sign In apps → profil applicatif")

  MyDigitalTwin — Apple — Résumé
  App installs :    3,923 → apple_app_installs.parquet
  Sign In apps :       62 → apple_signin_apps.parquet

  Usage ML :
  - App installs → K-Means temporel (axe 3)
  - App installs → centres d'intérêt ALS (axe 2)
  - Sign In apps → profil applicatif


In [13]:
spark.stop()